# PhishGuard AI: Reproducible Experiment Review

This notebook reads the prepared split labels, saved E1-E5 results, and generated figures. It does not retrain models or modify their weights. The URL and text corpora are independent and unpaired; E4 is an exploratory combination of real held-out rows, not naturally paired multimodal evaluation.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

# Work whether Jupyter starts in the repository root or notebooks/.
ROOT = Path.cwd()
if not (ROOT / 'models').is_dir():
    ROOT = ROOT.parent
DATA = ROOT / 'data' / 'processed'
DOCS = ROOT / 'documentation'
RESULTS_PATH = ROOT / 'models' / 'experiment_metrics.json'
with RESULTS_PATH.open(encoding='utf-8') as f:
    results = json.load(f)
print(f'Project root: {ROOT}')

## Prepared split class distributions
Read labels only to keep memory use low. Counts are read from the existing processed CSV files.

In [ ]:
distribution_rows = []
for modality in ('url', 'text'):
    for split in ('train', 'val', 'test'):
        labels = pd.read_csv(DATA / f'{modality}_{split}.csv', usecols=['label'])['label']
        counts = labels.value_counts().to_dict()
        distribution_rows.append({
            'Modality': modality, 'Split': split, 'Rows': int(len(labels)),
            'Safe (0)': int(counts.get(0, 0)), 'Phishing (1)': int(counts.get(1, 0)),
        })
split_distribution = pd.DataFrame(distribution_rows)
print(split_distribution.to_string(index=False))

## E1-E5 saved experiment results
E1 and E2 use their independent held-out test splits. E3 contains authored fixture checks only. E4 pairings are explicitly unpaired exploratory combinations; do not interpret them as paired multimodal performance.

In [ ]:
metric_rows = []
for name, metrics in (
    ('URL-only E1', results['E1_url_test']),
    ('Text-only E2', results['E2_text_test']),
    ('Controlled late fusion E4', results['E4_controlled_unpaired_pairs']['late_fusion_on_selected_rows']),
):
    metric_rows.append({'Evaluation': name, **{key: metrics[key] for key in
        ('accuracy', 'precision', 'recall', 'f1', 'false_positive_rate')},
        'Confusion matrix [[TN, FP], [FN, TP]]': metrics['confusion_matrix']})
print(pd.DataFrame(metric_rows).to_string(index=False))

fixture_rows = [{
    'Fixture': item['name'], 'Expected behavior': item['expected_structural_behavior'],
    'Risk score': item['risk_score'], 'Check passed': item['fixture_check_passed'],
} for item in results['E3_html_fixtures']]
print(pd.DataFrame(fixture_rows).to_string(index=False))

e4 = results['E4_controlled_unpaired_pairs']
print(pd.DataFrame([
    {'Scenario': name, **data}
    for name, data in e4['cross_class_scenarios'].items()
]).to_string(index=False))

error_rows = []
for modality, by_kind in results['E5_error_analysis'].items():
    for kind, details in by_kind.items():
        error_rows.append({'Modality': modality, 'Error': kind, 'Count': details['count']})
print(pd.DataFrame(error_rows).to_string(index=False))

## Charts
The charts are generated by `src/evaluate_experiments.py` from the held-out model predictions.

In [ ]:
for filename in ('model_comparison.png', 'confusion_matrices.png'):
    image_path = DOCS / 'charts' / filename
    if image_path.is_file():
        print(filename)
        image = plt.imread(image_path)
        plt.figure(figsize=(10, 5))
        plt.imshow(image)
        plt.axis('off')
        try:
            from IPython.display import display
            display(plt.gcf())  # Jupyter displays the figure inline.
        except ImportError:
            print('Install Jupyter to display this figure inline.')
        plt.close()
    else:
        print(f'Missing chart: {image_path}')

## Interpretation notes
No model weights are changed by this notebook. Fusion defaults are not empirically tuned. URL and text metrics come from separate source populations, and the HTML checks do not estimate HTML classification accuracy. See `documentation/experiment_results.md` for the complete protocol and error analysis.